# 1. Import & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 2. Daten einlesen

In [1]:
file_path = "../data/Number_of_fires_by_month.csv"
# CSV mit Semikolon-Trenner einlesen
df = pd.read_csv(file_path, sep=";", header=1, encoding="ISO-8859-1")

# 3. Datenbereinigung

In [ ]:
# Spaltennamen aufräumen
df.columns = df.columns.str.strip()

# "Data Qualifier"-Spalte entfernen (falls vorhanden)
df = df.loc[:, ~df.columns.str.contains('Data Qualifier', case=False)]


# Alle 'Unspecified'-Einträge in 'Month' löschen
df = df[df["Month"].notna()]
df = df[~df['Month'].str.contains("Unspecified", case=False, na=False)]


# Alle leeren Werte durch NaN ersetzen
df = df.replace(["", " ",], np.nan)


# "Wide" → "Long" Format
df_long = df.melt(
    id_vars=["Jurisdiction", "Month"],  # diese Spalten bleiben fix
    var_name="Year",                    # Name der neuen "Jahr"-Spalte
    value_name="Number_fires"                  # Name für die Werte (z. B. Anzahl Brände)
)

# 4. Explorative Visualisierung

Basisinfo

In [ ]:
print(df_long.info())
print(df_long.describe())

# Fehlende Werte je Spalte
print(df_long.isna().sum())


Zeitreihen Visualisierung

In [ ]:
# Gesamtzeitreihe
ts_total = (
    df_long.groupby(["Year", "Month"])["Number_fires"]
    .sum()
    .reset_index()
)
ts_total["Date"] = pd.to_datetime(ts_total["Year"].astype(str) + "-" + ts_total["Month"].astype(str))

plt.figure(figsize=(14,5))
plt.plot(ts_total["Date"], ts_total["Number_fires"])
plt.title("Total Fires per Month (Canada)")
plt.xlabel("Date")
plt.ylabel("Number of Fires")
plt.show()


Zeitreihen Jurisdiction

In [ ]:
plt.figure(figsize=(14,6))
for j in df_long["Jurisdiction_raw"].unique():
    subset = df_long[df_long["Jurisdiction_raw"] == j]
    plt.plot(
        pd.to_datetime(subset["Year"].astype(str) + "-" + subset["Month"].astype(str)),
        subset["Number_fires"],
        label=j,
        alpha=0.6
    )
plt.legend()
plt.title("Fires per Month by Jurisdiction")
plt.show()


Saisonale Muster

In [ ]:
monthly_avg = df_long.groupby("Month")["Number_fires"].mean()

plt.figure(figsize=(10,4))
monthly_avg.plot(kind="bar")
plt.title("Average Fires per Month (Seasonality)")
plt.xlabel("Month")
plt.ylabel("Avg Fires")
plt.show()


In [ ]:
plt.figure(figsize=(12,5))
df_long.boxplot(column="Number_fires", by="Month")
plt.title("Distribution of Fires per Month")
plt.suptitle("")
plt.show()

Heatmap Jahr x Monat

In [ ]:
pivot = df_long.pivot_table(
    index="Year",
    columns="Month",
    values="Number_fires",
    aggfunc="sum"
)

plt.figure(figsize=(12,6))
sns.heatmap(pivot, cmap="YlOrRd")
plt.title("Heatmap of Fires (Year × Month)")
plt.show()

Autokorrelation

In [ ]:
# Kanada-gesamt Zeitreihe
ts_total_sorted = ts_total.sort_values("Date")

plot_acf(ts_total_sorted["Number_fires"].dropna(), lags=40)
plt.show()

plot_pacf(ts_total_sorted["Number_fires"].dropna(), lags=40)
plt.show()

# 5. Feature Engineering

In [ ]:
# Datentypen anpassen
df_long["Year"] = df_long["Year"].astype(int)

# Anzahl (Number_fires) als ganze Zahl
df_long["Number_fires"] = pd.to_numeric(df_long["Number_fires"], errors="coerce").astype("Int64")

# Jurisdiction als Kategorie, Month als 1–12 umwandeln
df_long["Jurisdiction"] = df_long["Jurisdiction"].astype("category")
df_long["Month"] = pd.to_datetime(df_long["Month"], format="%B").dt.month

# Funktion zur Zuweisung der Jahreszeit (Season)
def get_season(month):
    # Definitionen: Frühling (März-Mai), Sommer (Juni-August),
    # Herbst (September-November) und Winter (Dezember-Februar)
    if month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    elif month in [9, 10, 11]:
        return "Fall"
    elif month in [12, 1, 2]:
        return "Winter"
    return np.nan

# Anwenden der Funktion auf die numerische Monatsspalte
df_long["Season"] = df_long["Month"].apply(get_season)

# Lag hinzufügen, vorherige Werte merken
df_long = df_long.sort_values(["Jurisdiction", "Year", "Month"])

df_long["Lag_1"] = df_long.groupby("Jurisdiction")["Number_fires"].shift(1)
df_long["Lag_2"] = df_long.groupby("Jurisdiction")["Number_fires"].shift(2)
df_long["Lag_3"] = df_long.groupby("Jurisdiction")["Number_fires"].shift(3)

# One Hot Encoding
df_long["Jurisdiction_raw"] = df_long["Jurisdiction"].astype(str)
df_long = pd.get_dummies(df_long, columns=["Jurisdiction"], prefix="REG", drop_first=False)
reg_cols = [col for col in df_long.columns if col.startswith("REG")]
df_long[reg_cols] = df_long[reg_cols].astype(int)

# Kontrolle
print(df_long.head(20))
print(df_long.dtypes)

In [ ]:
#Datei speichern
output_path = "../data/cleaned_Number_of_fires_by_month.csv"
df_long.to_csv(output_path, index=False)

# 6. Modellauswahl

# 7. Training & Evaluation

# 8. Interpretation & Ergebnisse

# 9. Export der Modelle/Plots